# Exploration — find the question

This notebook is scratch work. Anything that turns out to matter gets moved into a module in `flighttrack/` and tested there.

Run `python -m flighttrack.analyze` first so `data/processed/` exists, then work through the cells. The goal of this notebook is one sentence that starts with **"I noticed that…"**

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import flighttrack` works from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

fixes = pd.read_parquet("../data/processed/flights.parquet")
metrics = pd.read_csv("../data/processed/metrics.csv", parse_dates=["start_time", "end_time"])
print(f"{len(fixes):,} fixes, {fixes['icao24'].nunique():,} aircraft, {len(metrics):,} flights")

## 1. Look before you compute

`.head()`, `.describe()`, `.isna().sum()` — the three calls that tell you what you are actually holding.

In [ ]:
fixes.head()

In [ ]:
fixes.isna().sum()

In [ ]:
metrics[["n_points", "duration_min", "track_distance_km", "gc_distance_km", "detour_ratio", "max_altitude_ft", "mean_speed_kt"]].describe().round(2)

## 2. The interesting number

Detour ratio = distance actually flown ÷ straight-line distance between first and last fix. Which flights are the outliers, and why?

In [ ]:
metrics.sort_values("detour_ratio", ascending=False).head(10)[["flight_id", "callsign", "n_points", "duration_min", "track_distance_km", "gc_distance_km", "detour_ratio", "max_altitude_ft"]]

In [ ]:
# Plot one suspicious flight to see what it actually did.
worst = metrics.sort_values("detour_ratio", ascending=False).iloc[0]["flight_id"]
one = fixes[fixes["flight_id"] == worst]
plt.plot(one["longitude"], one["latitude"], marker=".")
plt.title(worst)
plt.xlabel("Longitude (°)"); plt.ylabel("Latitude (°)")

## 3. Questions to poke at

- Does the detour ratio depend on how high the aircraft cruises? (`max_altitude_ft` vs `detour_ratio`)
- Do aircraft from one `origin_country` cluster at a particular altitude?
- Is there a time of day when more aircraft are climbing than descending? (`vertical_rate_fpm` by hour)
- How does the flight count change with `--gap-minutes`? Is 30 the right threshold for this region?

In [ ]:
plt.scatter(metrics["max_altitude_ft"], metrics["detour_ratio"], s=8, alpha=0.5)
plt.ylim(0.95, 2)
plt.xlabel("Max altitude (ft)"); plt.ylabel("Detour ratio")

## 4. I noticed that…

_(write the one sentence here, then go build the chart that shows it)_